In [9]:
!pip install gymnasium
from collections import defaultdict
import gymnasium as gym # Changed to gymnasium
import numpy as np

# Create the Blackjack environment
env = gym.make('Blackjack-v1')

In [10]:

# -----------------------------------------------------------------------------
# 1. Helper Function: Generate Episode
# -----------------------------------------------------------------------------
def generate_episode(env, policy):
    """Generates a single episode from the environment using a given policy."""
    episode = []
    # For Gymnasium, env.reset() returns (observation, info).
    # The Blackjack-v1 observation is a 3-tuple (player_sum, dealer_card, usable_ace).
    # We need to unpack the observation from the reset return.
    state, _ = env.reset()
    while True:
        action = policy(state)
        next_state, reward, terminated, truncated, _ = env.step(action) # Correct for new_step_api/gymnasium
        done = terminated or truncated # Combine terminated and truncated for 'done'
        episode.append((state, action, reward))
        if done:
            break
        state = next_state
    return episode


In [11]:
def mc_prediction_q(env, num_episodes, policy, gamma=1.0):
    """Evaluates a policy using First-Visit Monte Carlo action-value prediction."""
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)
    Q = defaultdict(lambda: np.zeros(env.action_space.n))

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        states_actions_returns = []
        G = 0.0

        # Calculate returns backwards
        for state, action, reward in reversed(episode):
            G = gamma * G + reward
            states_actions_returns.append((state, action, G))

        states_actions_returns.reverse()

        # First-visit check
        visited = set()
        for state, action, G in states_actions_returns:
            if (state, action) not in visited:
                visited.add((state, action))
                returns_sum[(state, action)] += G
                returns_count[(state, action)] += 1.0
                Q[state][action] = returns_sum[(state, action)] / returns_count[(state, action)]

    return Q

In [12]:
def make_epsilon_greedy_policy(Q, epsilon, nA):
    """Creates an epsilon-greedy policy function based on Q-values."""
    def policy_fn(state):
        A = np.ones(nA, dtype=float) * (epsilon / nA)
        # Q[state] might be empty if the state has not been visited yet
        if state not in Q or np.all(Q[state] == 0):
            best_action = np.random.choice(nA) # Choose a random action if no Q-values are known
        else:
            best_action = np.argmax(Q[state])
        A[best_action] += (1.0 - epsilon)
        return np.random.choice(np.arange(nA), p=A)
    return policy_fn

def mc_control_epsilon_greedy(env, num_episodes, gamma=1.0, epsilon=0.1):
    """Performs on-policy first-visit MC control with an epsilon-greedy policy."""
    nA = env.action_space.n
    Q = defaultdict(lambda: np.zeros(nA))
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)

    # Initial policy
    policy = make_epsilon_greedy_policy(Q, epsilon, nA)

    for i in range(1, num_episodes + 1):
        # Generate an episode using the current epsilon-greedy policy
        episode = generate_episode(env, policy)

        visited = set()
        G = 0.0
        for state, action, reward in reversed(episode):
            G = gamma * G + reward
            if (state, action) not in visited:
                visited.add((state, action))
                returns_sum[(state, action)] += G
                returns_count[(state, action)] += 1.0
                Q[state][action] = returns_sum[(state, action)] / returns_count[(state, action)]

        # Update policy greedily with respect to newly updated Q values
        policy = make_epsilon_greedy_policy(Q, epsilon, nA)

    return Q, policy

# Example execution: Run MC control for 500,000 episodes
optimized_Q, optimized_policy = mc_control_epsilon_greedy(env, num_episodes=500000)

In [13]:
sample_states = [
    (14, 7, False),  # Hard 14 against a 7
    (19, 10, False), # Hard 19 against a 10
    (15, 2, True)    # Soft 15 against a 2
]

print("--- MC CONTROL RESULTS ---")
for state in sample_states:
    action_values = optimized_Q[state]
    best_action = np.argmax(action_values)
    action_name = "HIT" if best_action == 1 else "STICK"

    print(f"State {state}:")
    print(f"  Q-values (STICK, HIT): {action_values}")
    print(f"  Optimal Action: {action_name}\n")

--- MC CONTROL RESULTS ---
State (14, 7, False):
  Q-values (STICK, HIT): [-0.46557377 -0.36279309]
  Optimal Action: HIT

State (19, 10, False):
  Q-values (STICK, HIT): [-0.00842678 -0.7064803 ]
  Optimal Action: STICK

State (15, 2, True):
  Q-values (STICK, HIT): [-0.3030303   0.02422907]
  Optimal Action: HIT

